In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import os
from natsort import natsorted

import scanpy as sc
import seaborn as sns

from scroutines import basicu

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tools.sm_exceptions import ValueWarning
from tqdm import tqdm


import sys
sys.path.insert(0, '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/myvisctx/analysis_multiome/')
import lmm

In [2]:
outfigdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/'
f = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/superdupermegaRNA_hasraw_multiome_P21NRDR.h5ad'
adata = sc.read(f)
adata

AnnData object with n_obs × n_vars = 31039 × 16572
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden'
    var: 'feature_types'

In [3]:
adata = adata[adata.obs['Subclass'].isin(['L2/3', 'L4', 'L5IT', 'L6IT'])]
print(adata.obs['Subclass'].unique())
adata

['L2/3', 'L4', 'L6IT', 'L5IT']
Categories (4, object): ['L2/3', 'L4', 'L5IT', 'L6IT']


View of AnnData object with n_obs × n_vars = 15329 × 16572
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden'
    var: 'feature_types'

In [4]:
print(adata.X.data)
print(adata.raw.X.data)

[0.38399825 0.38399825 1.0552076  ... 0.47486174 0.47486174 2.4799154 ]
[35.  2.  1. ...  1.  4.  1.]


In [5]:
adata.obs['Age'].unique()

['P21', 'P21DR']
Categories (2, object): ['P21', 'P21DR']

In [6]:
adata.obs['Sample'].unique()

['P21a', 'P21b', 'P21DRa', 'P21DRb']
Categories (4, object): ['P21DRa', 'P21DRb', 'P21a', 'P21b']

In [7]:
adata.obs['total_counts'].unique()

array([21809., 26366., 14682., ..., 15104.,  4731., 16848.], dtype=float32)

In [8]:
import time

In [9]:
%%time

obs_fixed1 = 'Age'
obs_fixed2 = None # 'Light'
obs_random = 'Sample'

offset = 1e-2
scale = 1e4

for cluster in [
    # 'L2/3', 
    'L4', 'L5IT', 'L6IT']:
    tag = f"d260303_{cluster.replace('/', '')}"
    output = os.path.join(outfigdir, f'NRDR_DEGs_LMM_yoo25_P21_{tag}.csv')

    adatasub = adata[adata.obs['Subclass']==cluster]
    genes = adatasub.var.index.values 

    if obs_fixed2 is None:
        obs = adatasub.obs[[obs_fixed1, obs_random]].copy()
    else:
        obs = adatasub.obs[[obs_fixed1, obs_fixed2, obs_random]].copy()
    obs = obs.dropna()
    adatasub = adatasub[obs.index]

    # mat_raw = np.array(adatasub.X.todense())
    mat_raw = np.array(adatasub.raw.X.todense())
    
    # ### test
    # adatasub = adatasub[:,:20]
    # genes = genes[:20]
    # mat_raw = mat_raw[:,:20]
    # ### test

    # mat (CP10k norm)
    # mat = mat_raw/adatasub.obs['n_counts'].values.reshape(-1,1)*scale
    mat = mat_raw/adatasub.obs['total_counts'].values.reshape(-1,1)*scale

    res = lmm.run_lmm(mat, genes, obs, obs_fixed1, obs_random, output_csv=output, offset=offset)
    print(output)

(5931, 16572) (5931, 2)
(5931, 16522) (5931, 2)
(5931, 8777) (5931, 2)
133 ['Zdbf2' 'Ptprn' 'Gm28294' 'Bcl2' 'Btg2' 'Glul' 'Atp1a2' 'Camk1g' 'Pfkfb3'
 'Arl5b' 'Ptgds' 'Rnd3' 'Nr4a2' 'Tanc1' 'Bdnf' 'Cst3' 'Cbln4' 'Rps21'
 'Rpl39' 'Xist' 'Plp1' 'Car2' 'Skil' 'Tiparp' 'Bcan' 'S100a1' 'S100a16'
 'Gstm5' 'Gstm1' 'Ddit4l' 'Gm17501' 'Gm12371' 'Tgfbr1' 'Nr4a3' 'Adamtsl1'
 'Ak4' 'Stk40' 'Tnfrsf25' 'Fosl2' 'Mn1' 'Sgsm1' 'Tsc22d4' 'Pon2' 'Ptprz1'
 'B4galnt3' 'Prkd2' 'Fosb' 'Apoe' 'Nfkbid' 'Mag' 'Akap13' 'Mapk3' 'Egr2'
 'Fam13c' 'Midn' 'Gadd45b' 'Nab2' 'Cd63' 'Irs2' 'Prag1' 'Junb' 'Mt3' 'Mt2'
 'Mt1' 'Cdyl2' 'Cbfa2t3' 'Fam107a' 'Anxa11' 'Ndrg2' 'Clu' 'Egr3' 'Clmp'
 'Sik2' 'Gm17231' 'Arid3b' 'Smad3' 'Trim71' 'Gm2a' 'Sparc' 'Kdm6b'
 '1700016P03Rik' 'Aldoc' 'Fmnl1' 'Map3k14' 'Hist1h2bc' 'Gfod1' 'Dok3'
 'Gm47423' 'Pcsk1' 'Homer1' 'Rps29' 'Frmd6' 'Inf2' 'Selenop' 'Rpl37'
 'Slc1a3' 'Klf10' 'Zhx2' 'Trib1' 'Arc' 'Csdc2' 'Ccdc134' 'Phf21b' 'Mlc1'
 'Grasp' 'Etv5' 'Arhgap31' 'Plcxd2' 'Synj2' 'Sox8' 'Dusp1' 'G

100% 8777/8777 [08:14<00:00, 17.74it/s]


71 ['Zdbf2' 'Ptprn' 'Gm28294' 'Bcl2' 'Btg2' 'Camk1g' 'Rnd3' 'Nr4a2' 'Tanc1'
 'Bdnf' 'Cbln4' 'Xist' 'Skil' 'Tiparp' 'Gm17501' 'Gm12371' 'Tnfrsf25'
 'Fosl2' 'Sgsm1' 'Tsc22d4' 'B4galnt3' 'Prkd2' 'Fosb' 'Nfkbid' 'Akap13'
 'Mapk3' 'Egr2' 'Fam13c' 'Midn' 'Gadd45b' 'Nab2' 'Irs2' 'Prag1' 'Cdyl2'
 'Cbfa2t3' 'Anxa11' 'Egr3' 'Clmp' 'Sik2' 'Arid3b' 'Smad3' 'Gm2a'
 '1700016P03Rik' 'Fmnl1' 'Map3k14' 'Gfod1' 'Dok3' 'Pcsk1' 'Homer1' 'Frmd6'
 'Inf2' 'Klf10' 'Zhx2' 'Csdc2' 'Ccdc134' 'Phf21b' 'Grasp' 'Etv5'
 'Arhgap31' 'Plcxd2' 'Synj2' 'Grm4' 'Cdkn1a' 'Sik1' 'Plekhh2' 'Kdm5d'
 'Eif2s3y' 'Uty' 'Ddx3y' 'Aldh1a1' 'Dusp5']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_L4.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_L4.csv
(1211, 16572) (1211, 2)
(1211, 16205) (1211, 2)
(1211, 9075) (1211, 2)
96 ['Zdbf2' 'Ikzf2' 'Gm28294' 'Cdh7' 'Btg2' 'Apobec4' 'Ier5' 'Atp1a2'
 'Camk1g' 'Ptgds' 'Tnfaip6' 'Nr4a2' 'G

100% 9075/9075 [10:53<00:00, 13.88it/s]


46 ['Cdh7' 'Btg2' 'Apobec4' 'Tnfaip6' 'Mamld1' 'Xist' '4921511C10Rik'
 'Gm17501' 'Gm11802' 'Calb1' 'Gpr3' 'Rheb' 'Fosl2' 'Kctd8' 'Sgsm1' 'Mest'
 'Fosb' 'Mag' 'Dock1' 'Fam229b' 'Egr2' 'Midn' 'Gadd45b' 'Pawr' 'Phlda1'
 'Irs2' 'Mast3' 'Cbfa2t3' 'Slc25a37' 'Per1' '1700016P03Rik' 'Faap100'
 'Dok3' 'Pcsk1' 'Homer1' 'Trib1' 'Phf21b' 'Plcxd2' 'Col8a1' 'Lix1' 'Sik1'
 'Cntnap5c' 'Kdm5d' 'Uty' 'Ddx3y' 'Dusp5']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_L5IT.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_L5IT.csv
(1718, 16572) (1718, 2)
(1718, 16279) (1718, 2)
(1718, 9211) (1718, 2)
119 ['Zdbf2' 'Rassf5' 'Etnk2' 'Colgalt2' 'Apobec4' 'Camk1g' 'Pfkfb3' 'Arl5b'
 'Ptgds' 'Fam129b' 'Tnfaip6' 'Rtn4rl2' 'Nnat' 'Cdh22' 'Cbln4' 'Bcor'
 'Nsdhl' 'F8a' 'Xist' 'Tiparp' 'Gm6602' 'Penk' 'Tgfbr1' 'Lpar1' 'Ak4'
 '1700047F07Rik' 'Lrp8os2' 'Plk3' 'Foxo6' 'Stk40' 'Serinc2' 'Lypla2'
 'Zfp46' 'Disp3' 'Tnfrsf

100% 9211/9211 [24:30<00:00,  6.27it/s]


10 ['Xist' 'Tiparp' 'Sgsm1' 'Mast3' 'Sik2' 'Arid3b' '1700016P03Rik' 'Homer1'
 'Eif2s3y' 'Uty']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_L6IT.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_L6IT.csv
CPU times: user 43min 37s, sys: 16.8 s, total: 43min 54s
Wall time: 43min 49s
